# import libs

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from xgboost import XGBClassifier
import numpy as np
import joblib
import json

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

In [3]:
X_train = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/X_train.csv")
X_test  = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/X_test.csv")
X_oot   = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/X_oot.csv")

y_train = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/y_train.csv").squeeze()
y_test  = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/y_test.csv").squeeze()
y_oot   = pd.read_csv("R:/some-research-about-approve-model-in-banking/data/y_oot.csv").squeeze()

In [4]:
encoders   = joblib.load('R:/some-research-about-approve-model-in-banking/models/cat_encoders.pkl')
process    = joblib.load('R:/some-research-about-approve-model-in-banking/models/binning_process.pkl')

with open('R:/some-research-about-approve-model-in-banking/models/xgb_selected_features.json') as f:
    selected = json.load(f)

In [5]:
cat = X_train.select_dtypes(include=['object']).columns.tolist()

encoders = {}
for col in cat:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    encoders[col] = le

joblib.dump(encoders, 'R:/some-research-about-approve-model-in-banking/models/cat_encoders.pkl')
print(f"Encoded {len(cat)} categorical columns: {cat}")

Encoded 16 categorical columns: ['name_contract_type', 'code_gender', 'flag_own_car', 'flag_own_realty', 'name_type_suite', 'name_income_type', 'name_education_type', 'name_family_status', 'name_housing_type', 'occupation_type', 'weekday_appr_process_start', 'organization_type', 'fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']


In [6]:
for col in cat:
    X_test[col] = encoders[col].transform(X_test[col].astype(str))
    X_oot[col]  = encoders[col].transform(X_oot[col].astype(str))

In [7]:
train_with_label = X_train.copy()
train_with_label['target'] = y_train.values

test_with_label = X_test.copy()
test_with_label['target'] = y_test.values

oot_with_label = X_oot.copy()
oot_with_label['target'] = y_oot.values

In [8]:
for segment in train_with_label['flag_own_car'].unique():
    seg_data = train_with_label[train_with_label['flag_own_car'] == segment]
    print(f"{segment}: n={len(seg_data)}, default_rate={seg_data['target'].mean():.3f}")

0: n=138105, default_rate=0.085
1: n=71002, default_rate=0.073


# training with full dataset and try with segmentation by flag_own_car

In [9]:
def calc_gini(y_true, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    return 2 * auc - 1

In [21]:
len(X_train['flag_own_car'].unique())

2

In [23]:
full_model = XGBClassifier(max_depth=4)

full_model.fit(X_train[selected],y_train)

gini_train = calc_gini(y_train, full_model.predict_proba(X_train[selected])[:,1])
gini_test  = calc_gini(y_test,  full_model.predict_proba(X_test[selected])[:,1])
gini_oot   = calc_gini(y_oot,   full_model.predict_proba(X_oot[selected])[:,1])

print(f"full dataset:")
print(f"default rate train set: {round(y_train.mean(),5)}")
print(f"default rate test set: {round(y_test.mean(), 5)}")
print(f"default rate oot set: {round(y_oot.mean(), 5)}")

print("-"*10)
print(f"gini_train: {round(gini_train, 5)}")
print(f"gini_test: {round(gini_test, 5)}")
print(f"gini_oot: {round(gini_oot, 5)}")
print(f"metric drop: {round(1- gini_oot/gini_train,5)} from gini_train = {round(gini_train, 5)}")
print(f"{'-'*50}")

for i in X_train['flag_own_car'].unique():
    sub_model = XGBClassifier(max_depth=4)
    
    train = train_with_label[train_with_label['flag_own_car'] == i]
    test  = test_with_label[test_with_label['flag_own_car'] == i]
    oot   = oot_with_label[oot_with_label['flag_own_car'] == i]
    
    if len(train['flag_own_car'].unique()) == 1:

        x_train = train.drop('target', axis=1)
        y_train_ = train['target']
        x_test  = test.drop('target', axis=1)
        y_test_  = test['target']
        x_oot   = oot.drop('target', axis=1)
        y_oot_   = oot['target']

        train[selected + ['target']].to_csv(f"R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/train_flag_own_car_{i}.csv")
        test[selected + ['target']].to_csv(f"R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/test_flag_own_car_{i}.csv")
        oot[selected + ['target']].to_csv(f"R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/oot_flag_own_car_{i}.csv")
        print("done split and export data")

        sub_model.fit(x_train[selected], y_train_)

        gini_train = calc_gini(y_train_, sub_model.predict_proba(x_train[selected])[:,1])
        gini_test  = calc_gini(y_test_,  sub_model.predict_proba(x_test[selected])[:,1])
        gini_oot   = calc_gini(y_oot_,   sub_model.predict_proba(x_oot[selected])[:,1])

        print(f"segment name_housing_type = {i}:")
        print(f"default rate train set: {round(y_train_.mean(), 5)}")
        print(f"default rate test set: {round(y_test_.mean(), 5)}")
        print(f"default rate oot set: {round(y_oot_.mean(), 5)}")

        print("-"*10)
        print(f"gini_train: {round(gini_train, 5)}")
        print(f"gini_test: {round(gini_test,5)}")
        print(f"gini_oot: {round(gini_oot, 5)}")
        print(f"metric drop: {round(1- gini_oot/gini_train,5)} from gini_train = {round(gini_train, 5)}")
        print(f"{'-'*50}")
    else:
        print('split wrong')

full dataset:
default rate train set: 0.08101
default rate test set: 0.08101
default rate oot set: 0.07915
----------
gini_train: 0.62037
gini_test: 0.53212
gini_oot: 0.52632
metric drop: 0.1516 from gini_train = 0.62037
--------------------------------------------------
done split and export data
segment name_housing_type = 0:
default rate train set: 0.08508
default rate test set: 0.0863
default rate oot set: 0.08316
----------
gini_train: 0.64473
gini_test: 0.52737
gini_oot: 0.52603
metric drop: 0.18411 from gini_train = 0.64473
--------------------------------------------------
done split and export data
segment name_housing_type = 1:
default rate train set: 0.07308
default rate test set: 0.07071
default rate oot set: 0.07148
----------
gini_train: 0.73298
gini_test: 0.49024
gini_oot: 0.47487
metric drop: 0.35215 from gini_train = 0.73298
--------------------------------------------------


# explain about smaller sample → less variance → inflated AUC

when dividing a large dataset into smaller segments, the reduced sample size makes the model highly susceptible to noise

- This leads to a falsely high training metric that drops dramatically on the OOT set
- Ultimately, over-segmentation causes inadvertent overfitting—a classic trap of metric chasing